In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Energy_usage_Tracker').getOrCreate()

In [2]:
from google.colab import files
uploaded = files.upload ()

Saving devices.csv to devices.csv
Saving sensor_logs.csv to sensor_logs.csv


In [3]:
logs_df = spark.read.csv("sensor_logs.csv",header=True,inferSchema=True)

devices_df = spark.read.csv("devices.csv",header=True,inferSchema=True)

In [4]:
logs_df.show()
logs_df.printSchema()

+---------+-------+-------------------+----------+
|device_id|room_id|          timestamp|energy_kwh|
+---------+-------+-------------------+----------+
|     D001|   R001|2026-06-01 07:30:00|       2.5|
|     D002|   R001|2026-06-01 08:15:00|       0.8|
|     D003|   R002|2026-06-01 09:00:00|       1.2|
|     D001|   R001|2026-06-01 18:30:00|       3.8|
|     D004|   R003|2026-06-01 19:15:00|       0.6|
|     D005|   R002|2026-06-01 20:00:00|       4.5|
|     D001|   R001|2026-06-02 07:45:00|       2.8|
|     D002|   R001|2026-06-02 08:10:00|      NULL|
|     D003|   R002|2026-06-02 10:00:00|       1.5|
|     D005|   R002|2026-06-02 18:00:00|       5.2|
|     D004|   R003|2026-06-02 21:00:00|       0.7|
|     D002|   R001|2026-06-03 08:00:00|       0.9|
|     D003|   R002|2026-06-03 11:00:00|       1.3|
|     D001|   R001|2026-06-03 18:45:00|       4.1|
|     D005|   R002|2026-06-03 19:30:00|      NULL|
|     D004|   R003|2026-06-03 22:00:00|       0.5|
|     D001|   R001|2026-06-04 0

In [5]:
devices_df.show()
devices_df.printSchema()

+---------+---------------+-------------+
|device_id|    device_name|  device_type|
+---------+---------------+-------------+
|     D001|Air Conditioner|      Cooling|
|     D002|   Refrigerator|      Cooling|
|     D003|Washing Machine|    Appliance|
|     D004|     Television|Entertainment|
|     D005|   Water Heater|      Heating|
+---------+---------------+-------------+

root
 |-- device_id: string (nullable = true)
 |-- device_name: string (nullable = true)
 |-- device_type: string (nullable = true)



In [6]:
from pyspark.sql.functions import *
null_counts = logs_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in logs_df.columns
])
null_counts.show()

+---------+-------+---------+----------+
|device_id|room_id|timestamp|energy_kwh|
+---------+-------+---------+----------+
|        0|      0|        0|         3|
+---------+-------+---------+----------+



In [7]:
logs_df.filter(col("energy_kwh").isNull()).show()

+---------+-------+-------------------+----------+
|device_id|room_id|          timestamp|energy_kwh|
+---------+-------+-------------------+----------+
|     D002|   R001|2026-06-02 08:10:00|      NULL|
|     D005|   R002|2026-06-03 19:30:00|      NULL|
|     D003|   R002|2026-06-05 12:00:00|      NULL|
+---------+-------+-------------------+----------+



In [8]:
logs_df = logs_df.fillna({
    "energy_kwh": 0.0
})
logs_df.show()

+---------+-------+-------------------+----------+
|device_id|room_id|          timestamp|energy_kwh|
+---------+-------+-------------------+----------+
|     D001|   R001|2026-06-01 07:30:00|       2.5|
|     D002|   R001|2026-06-01 08:15:00|       0.8|
|     D003|   R002|2026-06-01 09:00:00|       1.2|
|     D001|   R001|2026-06-01 18:30:00|       3.8|
|     D004|   R003|2026-06-01 19:15:00|       0.6|
|     D005|   R002|2026-06-01 20:00:00|       4.5|
|     D001|   R001|2026-06-02 07:45:00|       2.8|
|     D002|   R001|2026-06-02 08:10:00|       0.0|
|     D003|   R002|2026-06-02 10:00:00|       1.5|
|     D005|   R002|2026-06-02 18:00:00|       5.2|
|     D004|   R003|2026-06-02 21:00:00|       0.7|
|     D002|   R001|2026-06-03 08:00:00|       0.9|
|     D003|   R002|2026-06-03 11:00:00|       1.3|
|     D001|   R001|2026-06-03 18:45:00|       4.1|
|     D005|   R002|2026-06-03 19:30:00|       0.0|
|     D004|   R003|2026-06-03 22:00:00|       0.5|
|     D001|   R001|2026-06-04 0

In [9]:
logs_df.groupBy(col("device_id")).agg(
    sum("energy_kwh").alias("total_energy_kwh"),
    avg("energy_kwh").alias("avg_energy_kwh")).show()

+---------+------------------+------------------+
|device_id|  total_energy_kwh|    avg_energy_kwh|
+---------+------------------+------------------+
|     D002|               2.7|             0.675|
|     D003|               5.1|              1.02|
|     D004|2.6999999999999997|0.6749999999999999|
|     D005|              19.5|               3.9|
|     D001|              19.5|              3.25|
+---------+------------------+------------------+



In [10]:
energy_df = logs_df.join(
    devices_df,
    on="device_id",
    how="inner"
)
energy_df.show()

+---------+-------+-------------------+----------+---------------+-------------+
|device_id|room_id|          timestamp|energy_kwh|    device_name|  device_type|
+---------+-------+-------------------+----------+---------------+-------------+
|     D001|   R001|2026-06-01 07:30:00|       2.5|Air Conditioner|      Cooling|
|     D002|   R001|2026-06-01 08:15:00|       0.8|   Refrigerator|      Cooling|
|     D003|   R002|2026-06-01 09:00:00|       1.2|Washing Machine|    Appliance|
|     D001|   R001|2026-06-01 18:30:00|       3.8|Air Conditioner|      Cooling|
|     D004|   R003|2026-06-01 19:15:00|       0.6|     Television|Entertainment|
|     D005|   R002|2026-06-01 20:00:00|       4.5|   Water Heater|      Heating|
|     D001|   R001|2026-06-02 07:45:00|       2.8|Air Conditioner|      Cooling|
|     D002|   R001|2026-06-02 08:10:00|       0.0|   Refrigerator|      Cooling|
|     D003|   R002|2026-06-02 10:00:00|       1.5|Washing Machine|    Appliance|
|     D005|   R002|2026-06-0

In [11]:
energy_df = energy_df.withColumn(
    "hour",
    hour(col("timestamp"))
)

In [13]:
energy_df = energy_df.withColumn(
    "usage_type",
    when(
        (col("hour") >= 18) &(col("hour") < 22), "Peak"
    ).otherwise("Off-Peak")
)
energy_df.show()

+---------+-------+-------------------+----------+---------------+-------------+----+----------+
|device_id|room_id|          timestamp|energy_kwh|    device_name|  device_type|hour|usage_type|
+---------+-------+-------------------+----------+---------------+-------------+----+----------+
|     D001|   R001|2026-06-01 07:30:00|       2.5|Air Conditioner|      Cooling|   7|  Off-Peak|
|     D002|   R001|2026-06-01 08:15:00|       0.8|   Refrigerator|      Cooling|   8|  Off-Peak|
|     D003|   R002|2026-06-01 09:00:00|       1.2|Washing Machine|    Appliance|   9|  Off-Peak|
|     D001|   R001|2026-06-01 18:30:00|       3.8|Air Conditioner|      Cooling|  18|      Peak|
|     D004|   R003|2026-06-01 19:15:00|       0.6|     Television|Entertainment|  19|      Peak|
|     D005|   R002|2026-06-01 20:00:00|       4.5|   Water Heater|      Heating|  20|      Peak|
|     D001|   R001|2026-06-02 07:45:00|       2.8|Air Conditioner|      Cooling|   7|  Off-Peak|
|     D002|   R001|2026-06-02 

In [14]:
print("Peak vs Off-Peak Usage")
energy_df.groupBy(
    "device_id",
    "device_name",
    "usage_type"
).agg(sum("energy_kwh").alias("total_energy_kwh")
).show()

Peak vs Off-Peak Usage
+---------+---------------+----------+------------------+
|device_id|    device_name|usage_type|  total_energy_kwh|
+---------+---------------+----------+------------------+
|     D005|   Water Heater|      Peak|              19.5|
|     D001|Air Conditioner|      Peak|11.799999999999999|
|     D004|     Television|      Peak|2.1999999999999997|
|     D003|Washing Machine|  Off-Peak|               5.1|
|     D004|     Television|  Off-Peak|               0.5|
|     D001|Air Conditioner|  Off-Peak| 7.699999999999999|
|     D002|   Refrigerator|  Off-Peak|               2.7|
+---------+---------------+----------+------------------+



In [15]:
print("Total energy consumed by each device:\n")
device_usage = energy_df.groupBy(
    "device_id",
    "device_name"
).agg(sum("energy_kwh").alias("total_energy_kwh"))
device_usage.show()

Total energy consumed by each device:

+---------+---------------+------------------+
|device_id|    device_name|  total_energy_kwh|
+---------+---------------+------------------+
|     D001|Air Conditioner|              19.5|
|     D003|Washing Machine|               5.1|
|     D002|   Refrigerator|               2.7|
|     D005|   Water Heater|              19.5|
|     D004|     Television|2.6999999999999997|
+---------+---------------+------------------+



In [16]:
top_devices = device_usage.orderBy(desc("total_energy_kwh"))
print("Top Energy Consuming Devices")
top_devices.show()

Top Energy Consuming Devices
+---------+---------------+------------------+
|device_id|    device_name|  total_energy_kwh|
+---------+---------------+------------------+
|     D001|Air Conditioner|              19.5|
|     D005|   Water Heater|              19.5|
|     D003|Washing Machine|               5.1|
|     D002|   Refrigerator|               2.7|
|     D004|     Television|2.6999999999999997|
+---------+---------------+------------------+



In [17]:
high_usage_devices = device_usage.filter(col("total_energy_kwh") > 10)

print("High Usage Devices")
high_usage_devices.show()

High Usage Devices
+---------+---------------+----------------+
|device_id|    device_name|total_energy_kwh|
+---------+---------------+----------------+
|     D001|Air Conditioner|            19.5|
|     D005|   Water Heater|            19.5|
+---------+---------------+----------------+



In [18]:
print("Energy Consumption Report")

print("\nTotal sensor records:", logs_df.count())
print("Total devices:", devices_df.count())

print("Total energy consumed (kWh):")
energy_df.agg(sum("energy_kwh")).show()

print("\nTop Energy Consuming Device:")
device_usage.orderBy(col("total_energy_kwh").desc()).show(1)

print("\nLowest Energy Consuming Device:")
device_usage.orderBy(col("total_energy_kwh").asc()).show(1)

print("\nHighest Energy Reading:")
energy_df.orderBy(
    col("energy_kwh").desc()
).select(
    "device_id",
    "device_name",
    "timestamp",
    "energy_kwh"
).show(1)

print("\nLowest Energy Reading:")
energy_df.orderBy(
    col("energy_kwh").asc()
).select(
    "device_id",
    "device_name",
    "timestamp",
    "energy_kwh"
).show(1)

print("\nTotal Energy By Device Type:")
energy_df.groupBy(
    "device_type"
).agg(sum("energy_kwh").alias("total_energy_kwh")
).orderBy(col("total_energy_kwh").desc()
).show()

print("\nTotal Energy By Device:")
device_usage.orderBy(col("total_energy_kwh").desc()).show()

print("\nPeak vs Off-Peak Usage:")
energy_df.groupBy(
    "device_id",
    "device_name",
    "usage_type"
).agg(sum("energy_kwh").alias("total_energy_kwh")
).show()

Energy Consumption Report

Total sensor records: 24
Total devices: 5
Total energy consumed (kWh):
+-----------------+
|  sum(energy_kwh)|
+-----------------+
|49.49999999999999|
+-----------------+


Top Energy Consuming Device:
+---------+---------------+----------------+
|device_id|    device_name|total_energy_kwh|
+---------+---------------+----------------+
|     D001|Air Conditioner|            19.5|
+---------+---------------+----------------+
only showing top 1 row

Lowest Energy Consuming Device:
+---------+-----------+------------------+
|device_id|device_name|  total_energy_kwh|
+---------+-----------+------------------+
|     D004| Television|2.6999999999999997|
+---------+-----------+------------------+
only showing top 1 row

Highest Energy Reading:
+---------+------------+-------------------+----------+
|device_id| device_name|          timestamp|energy_kwh|
+---------+------------+-------------------+----------+
|     D005|Water Heater|2026-06-02 18:00:00|       5.2|
+--

In [19]:
top_devices.write.mode("overwrite").option("header", True).csv("output/top_energy_devices")